# Saans - Chest X-ray TB Screening Model (DenseNet121)

Real end-to-end training notebook for the Saans X-ray vision service.
Everything here runs **inside Google Colab** - data is downloaded to the Colab VM,
not to your laptop.

> **Honest scope note:** Trained on adult TB CXR datasets as proof-of-concept; paediatric fine-tuning is the Phase-1 pilot roadmap item.

**Blocks**
1. Download data (NIH NLM Shenzhen + Montgomery)
2. Preprocess (RGB, 224x224, normalise, augment, 80/20 split)
3. Train (DenseNet121, ImageNet pretrained, two-phase fine-tune)
4. Evaluate (accuracy, precision, recall, confusion matrix, curves)
5. Save weights -> `saans_xray_densenet121.h5`
6. Grad-CAM (real gradients -> heatmap overlay)
7. Inference -> `{tb_probability, heatmap_image}`
8. Download the `.h5` and put it in `backend/models/`

**Before you run:** `Runtime > Change runtime type > T4 GPU`. On CPU this takes hours.

Nothing in this notebook is simulated. Every number printed comes from the model
you just trained on real films. If a step fails, it fails loudly.

In [ ]:
# Environment check - run this first.
import sys
import tensorflow as tf

print("Python     ", sys.version.split()[0])
print("TensorFlow ", tf.__version__)
print("Keras      ", tf.keras.__version__)

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print("GPU        ", [g.name for g in gpus])
else:
    print("GPU         NONE  ->  Runtime > Change runtime type > T4 GPU.")
    print("             Training on CPU will take hours instead of ~10 minutes.")

## BLOCK 1 - Download the data (inside Colab)

Primary source: the official NIH / U.S. National Library of Medicine TB chest
X-ray sets listed at
<https://lhncbc.nlm.nih.gov/LHC-publications/downloads/TuberculosisChestXrayImageDataSets.html>

* **Shenzhen (China set)** - `ChinaSet_AllFiles.zip` (~662 CXRs)
* **Montgomery County** - `MontgomeryCountyXRaySet.zip` (~138 CXRs)

Labels come from the NLM filename convention: the digit after the last underscore
is the label - `..._0.png` = **Normal**, `..._1.png` = **TB**.

If the NIH download fails, the cell prints the Kaggle mirror for whichever set is
missing - they are separate datasets, one link each:

* Shenzhen - <https://www.kaggle.com/datasets/raddar/tuberculosis-chest-xrays-shenzhen>
* Montgomery - <https://www.kaggle.com/datasets/raddar/tuberculosis-chest-xrays-montgomery>

Download the zip, upload it with the folder icon on the left, and **re-run this
same cell** - it looks for local zips before it tries the network, so it will pick
up whatever you uploaded.

In [ ]:
# BLOCK 1 - DOWNLOAD DATA
# Runs inside the Colab VM. Nothing touches your laptop.

import os
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

DATA_ROOT = Path("/content/saans_data")
RAW_DIR = DATA_ROOT / "raw"            # zips live here
EXTRACT_DIR = DATA_ROOT / "extracted"  # unzipped images live here
RAW_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

NIH_PAGE = "https://lhncbc.nlm.nih.gov/LHC-publications/downloads/TuberculosisChestXrayImageDataSets.html"

# For each dataset: the NLM zip name, the filename prefix its images use, the
# mirrors to try in order, and the Kaggle page to fall back to by hand.
DATASETS = {
    "shenzhen": {
        "zip": "ChinaSet_AllFiles.zip",
        "prefix": "CHNCXR",
        "urls": [
            "https://data.lhncbc.nlm.nih.gov/public/Tuberculosis-Chest-X-ray-Datasets/China-CXRSet/ChinaSet_AllFiles.zip",
            "https://openi.nlm.nih.gov/imgs/collections/ChinaSet_AllFiles.zip",
        ],
        "kaggle": "https://www.kaggle.com/datasets/raddar/tuberculosis-chest-xrays-shenzhen",
    },
    "montgomery": {
        "zip": "MontgomeryCountyXRaySet.zip",
        "prefix": "MCUCXR",
        "urls": [
            "https://data.lhncbc.nlm.nih.gov/public/Tuberculosis-Chest-X-ray-Datasets/Montgomery-County-CXR-Set/MontgomerySet.zip",
            "https://data.lhncbc.nlm.nih.gov/public/Tuberculosis-Chest-X-ray-Datasets/Montgomery-County-CXR-Set/MontgomeryCountyXRaySet.zip",
            "https://openi.nlm.nih.gov/imgs/collections/NLM-MontgomeryCXRSet.zip",
        ],
        "kaggle": "https://www.kaggle.com/datasets/raddar/tuberculosis-chest-xrays-montgomery",
    },
}


def fallback_message(keys):
    """Printed when a dataset cannot be fetched automatically."""
    lines = ["Download failed."]
    for key in keys:
        lines.append(
            f"  {key}: go to {DATASETS[key]['kaggle']}, download the zip manually, and upload it "
            "to Colab using the folder icon on the left."
        )
    return "\n".join(lines)


def extract_all_zips():
    """Unzip every zip we can see - downloaded or uploaded by hand - into EXTRACT_DIR."""
    candidates = (
        list(RAW_DIR.glob("*.zip"))
        + list(Path("/content").glob("*.zip"))
        + [p for p in Path("/content").glob("*/*.zip") if "saans_data" not in str(p)]
    )
    for zpath in sorted(set(candidates)):
        if not zipfile.is_zipfile(zpath):
            print(f"  skipping {zpath.name}: not a valid zip")
            continue
        marker = EXTRACT_DIR / (".done_" + zpath.name)
        if marker.exists():
            continue
        print(f"  extracting {zpath.name} ...")
        with zipfile.ZipFile(zpath) as zf:
            zf.extractall(EXTRACT_DIR)
        marker.write_text("ok")


def find_images(prefix):
    """
    CXR files on disk for one dataset, following the NLM naming convention.

    The Montgomery zip also ships hand-drawn lung masks under ManualMask/ with
    the SAME filenames as the films they belong to - excluding anything with
    'mask' in its path keeps those binary masks out of the training set.
    """
    return sorted(
        p
        for p in EXTRACT_DIR.rglob("*")
        if p.suffix.lower() in {".png", ".jpg", ".jpeg"}
        and p.name.upper().startswith(prefix)
        and p.stem.rsplit("_", 1)[-1] in {"0", "1"}
        and "mask" not in str(p).lower()
        and "__MACOSX" not in str(p)
    )


def download(url, dest):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (Saans training notebook)"})
    with urllib.request.urlopen(req, timeout=180) as resp, open(dest, "wb") as fh:
        total = int(resp.headers.get("Content-Length") or 0)
        done = 0
        while True:
            chunk = resp.read(1 << 20)
            if not chunk:
                break
            fh.write(chunk)
            done += len(chunk)
            tail = f" / {total / 1e6:.1f} MB" if total else ""
            print(f"\r    {done / 1e6:7.1f} MB{tail}", end="")
    print()
    if not zipfile.is_zipfile(dest):
        dest.unlink(missing_ok=True)
        raise RuntimeError("server returned something that is not a zip (an HTML error page?)")
    return dest


print("Checking for data already on this VM (downloaded earlier, or uploaded by you)...")
extract_all_zips()

missing = []
for key, spec in DATASETS.items():
    found = find_images(spec["prefix"])
    if found:
        print(f"  {key:11s} already present: {len(found)} images")
    else:
        missing.append(key)

for key in list(missing):
    spec = DATASETS[key]
    print(f"\nDownloading {key} ({spec['zip']}) from NIH NLM ...")
    dest = RAW_DIR / spec["zip"]
    for url in spec["urls"]:
        try:
            print(f"  trying {url}")
            download(url, dest)
            break
        except (urllib.error.URLError, urllib.error.HTTPError, RuntimeError, TimeoutError, OSError) as exc:
            print(f"  ! failed: {type(exc).__name__}: {exc}")
    extract_all_zips()
    n = len(find_images(spec["prefix"]))
    if n:
        missing.remove(key)
        print(f"  {key} ready: {n} images")

if missing:
    print()
    print(fallback_message(missing))
    print()
    print(f"(official NLM file list: {NIH_PAGE})")
    # If images did land but none carry the _0 / _1 label suffix, say so plainly
    # rather than leaving you guessing why the count is zero.
    stray = [p.name for p in EXTRACT_DIR.rglob("*") if p.suffix.lower() in {".png", ".jpg", ".jpeg"}][:5]
    if stray:
        print()
        print("Images were found, but none match the NLM naming convention. Labels come from the")
        print("filename: CHNCXR_0001_0.png = Normal, CHNCXR_0001_1.png = TB. Found instead:")
        for name in stray:
            print("   ", name)
    raise SystemExit

In [ ]:
# BLOCK 1 (cont.) - build the labelled index from whatever landed on disk.
import pandas as pd

rows = []
for key, spec in DATASETS.items():
    for path in find_images(spec["prefix"]):
        label = int(path.stem.rsplit("_", 1)[-1])  # 0 = Normal, 1 = TB
        rows.append({"path": str(path), "label": label, "dataset": key})

df = pd.DataFrame(rows).drop_duplicates(subset="path").reset_index(drop=True)
if df.empty:
    raise RuntimeError("No labelled CXR images found after extraction.\n" + fallback_message(list(DATASETS)))

CLASS_NAMES = ["Normal", "TB"]
print(f"Total images: {len(df)}")
print()
print(pd.crosstab(df["dataset"], df["label"].map({0: "Normal", 1: "TB"}), margins=True))

## BLOCK 2 - Preprocess

* decode to **RGB** (the source films are single-channel; DenseNet expects 3)
* resize to **224 x 224**
* **normalise** with `densenet.preprocess_input` - the exact scaling the ImageNet
  weights were trained with (torch mode: /255, then per-channel mean/std)
* **augment** the training split only: horizontal flip, +/-21 deg rotation, 10% zoom
* **80/20 stratified split**, seeded, so both splits keep the same TB ratio

The films are decoded once into a uint8 array cached in RAM (~800 images), so the
very large source PNGs are not re-decoded every epoch.

In [ ]:
# BLOCK 2 - PREPROCESS
import numpy as np
import tensorflow as tf
from PIL import Image
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications.densenet import preprocess_input
from tqdm.auto import tqdm

IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42

tf.keras.utils.set_random_seed(SEED)


def load_uint8(path):
    with Image.open(path) as im:
        return np.asarray(im.convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR), dtype=np.uint8)


X = np.stack([load_uint8(p) for p in tqdm(df["path"], desc="decoding")])
y = df["label"].to_numpy().astype("float32")
print("X:", X.shape, X.dtype, " y:", y.shape, " TB fraction:", round(float(y.mean()), 3))

idx_train, idx_val = train_test_split(
    np.arange(len(y)), test_size=0.20, random_state=SEED, stratify=y
)
X_train, y_train = X[idx_train], y[idx_train]
X_val, y_val = X[idx_val], y[idx_val]
df_val = df.iloc[idx_val].reset_index(drop=True)  # keep the paths for Grad-CAM later

print(f"train: {len(y_train)}  (TB {int(y_train.sum())} / Normal {int((1 - y_train).sum())})")
print(f"val:   {len(y_val)}  (TB {int(y_val.sum())} / Normal {int((1 - y_val).sum())})")

augment = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal", seed=SEED),
        tf.keras.layers.RandomRotation(0.06, fill_mode="nearest", seed=SEED),  # ~ +/-21 degrees
        tf.keras.layers.RandomZoom(0.10, fill_mode="nearest", seed=SEED),
    ],
    name="augmentation",
)


def make_dataset(images, labels, training):
    ds = tf.data.Dataset.from_tensor_slices((images, labels))
    if training:
        ds = ds.shuffle(len(images), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE)

    def prep(xb, yb):
        xb = tf.cast(xb, tf.float32)
        if training:
            xb = augment(xb, training=True)
        return preprocess_input(xb), yb

    return ds.map(prep, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)


train_ds = make_dataset(X_train, y_train, training=True)
val_ds = make_dataset(X_val, y_val, training=False)  # unshuffled: predictions stay aligned with y_val
print(train_ds.element_spec)

In [ ]:
# BLOCK 2 (cont.) - eyeball a real augmented training batch.
import matplotlib.pyplot as plt

batch = next(iter(train_ds))[0].numpy()
# undo the densenet normalisation, for display only
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
show = np.clip(batch[:8] * std + mean, 0, 1)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, img in zip(axes.ravel(), show):
    ax.imshow(img)
    ax.axis("off")
fig.suptitle("Augmented training batch (real films from the NLM sets)")
plt.tight_layout()
plt.show()

## BLOCK 3 - Train DenseNet121 (ImageNet pretrained)

Two-phase transfer learning on ~800 films:

1. **Head only** - whole backbone frozen, train the new binary head (Adam 1e-3).
   Gets the randomly-initialised head somewhere sane without wrecking the
   pretrained features.
2. **Fine-tune the top** - unfreeze the last dense block (`conv5_*`) at Adam 1e-5.
   Early layers stay frozen (generic edge/texture filters) and every BatchNorm
   stays frozen, because the batches are small.

Binary cross-entropy with class weights. Target: **>= 85% validation accuracy**.

In [ ]:
# BLOCK 3 - TRAIN
# Trained on adult TB CXR datasets as proof-of-concept; paediatric fine-tuning
# is the Phase-1 pilot roadmap item.
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import callbacks, layers, models, optimizers

inputs = layers.Input((IMG_SIZE, IMG_SIZE, 3), name="cxr_input")
base = tf.keras.applications.DenseNet121(include_top=False, weights="imagenet", input_tensor=inputs)
x = layers.GlobalAveragePooling2D(name="gap")(base.output)
x = layers.Dropout(0.3, name="drop")(x)
outputs = layers.Dense(1, activation="sigmoid", name="tb_output")(x)
model = models.Model(inputs, outputs, name="saans_xray_densenet121")

METRICS = [
    "accuracy",
    tf.keras.metrics.AUC(name="auc"),
    tf.keras.metrics.Precision(name="precision"),
    tf.keras.metrics.Recall(name="recall"),
]

class_weight = dict(
    enumerate(compute_class_weight("balanced", classes=np.array([0.0, 1.0]), y=y_train))
)
print("class weights:", {int(k): round(float(v), 3) for k, v in class_weight.items()})

# ---- Phase 1: frozen backbone, train the head -----------------------------
base.trainable = False
model.compile(optimizer=optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=METRICS)
print(f"trainable params (phase 1): {int(sum(np.prod(w.shape) for w in model.trainable_weights)):,}")

hist1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weight,
    callbacks=[
        callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=4, restore_best_weights=True)
    ],
    verbose=1,
)

In [ ]:
# BLOCK 3 (cont.) - Phase 2: unfreeze the last dense block and fine-tune.
FINE_TUNE_FROM = "conv5_block1_0_bn"

base.trainable = True
names = [l.name for l in base.layers]
if FINE_TUNE_FROM not in names:
    raise RuntimeError(f"layer {FINE_TUNE_FROM} not found; conv5 layers are: {[n for n in names if n.startswith('conv5')][:5]}")
cut = names.index(FINE_TUNE_FROM)

for layer in base.layers[:cut]:
    layer.trainable = False
for layer in base.layers[cut:]:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False  # small batches -> keep the pretrained BN statistics

model.compile(optimizer=optimizers.Adam(1e-5), loss="binary_crossentropy", metrics=METRICS)
print(f"unfrozen from layer {cut}/{len(names)} ({FINE_TUNE_FROM})")
print(f"trainable params (phase 2): {int(sum(np.prod(w.shape) for w in model.trainable_weights)):,}")

hist2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    class_weight=class_weight,
    callbacks=[
        callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=7, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3, min_lr=1e-7, verbose=1),
    ],
    verbose=1,
)

history = {
    k: list(hist1.history.get(k, [])) + list(hist2.history.get(k, []))
    for k in hist2.history
    if k in hist1.history
}
best_val_acc = max(history["val_accuracy"])
print(f"\nBest validation accuracy: {best_val_acc:.4f}")
if best_val_acc >= 0.85:
    print("Target of 0.85 met.")
else:
    print(
        "Target of 0.85 NOT met - do not ship these weights. Options: run phase 2 longer, "
        "unfreeze from conv4_block1_0_bn instead, or turn the augmentation up."
    )

## BLOCK 4 - Evaluate

Accuracy, precision, recall, ROC-AUC, confusion matrix, and a saved training-curve
plot. All of it computed on the held-out 20% split, using the best weights that
`EarlyStopping` restored.

In [ ]:
# BLOCK 4 - EVALUATE
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)

eval_metrics = model.evaluate(val_ds, verbose=0, return_dict=True)
print("keras evaluate:", {k: round(float(v), 4) for k, v in eval_metrics.items()})

y_prob = model.predict(val_ds, verbose=0).ravel()
THRESHOLD = 0.5
y_pred = (y_prob >= THRESHOLD).astype(int)
y_true = y_val.astype(int)

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
auc = roc_auc_score(y_true, y_prob)

print(f"\nValidation set: {len(y_true)} films  (threshold {THRESHOLD})")
print(f"  accuracy : {acc:.4f}")
print(f"  precision: {prec:.4f}")
print(f"  recall   : {rec:.4f}   <- the one that matters for a screening tool")
print(f"  roc auc  : {auc:.4f}")
print()
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

cm = confusion_matrix(y_true, y_pred)
print("Confusion matrix (rows = true, cols = predicted):")
print(pd.DataFrame(cm, index=[f"true {c}" for c in CLASS_NAMES], columns=[f"pred {c}" for c in CLASS_NAMES]))
tn, fp, fn, tp = cm.ravel()
print(f"\nMissed TB (false negatives): {fn}    False alarms (false positives): {fp}")

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Saans CXR - validation confusion matrix")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
# BLOCK 4 (cont.) - training curves, saved to training_curves.png
n1 = len(hist1.history["loss"])  # epoch where phase 2 starts

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, key, title in zip(axes, ["accuracy", "loss", "auc"], ["Accuracy", "Loss", "ROC AUC"]):
    ax.plot(history[key], label=f"train {key}")
    ax.plot(history["val_" + key], label=f"val {key}")
    ax.axvline(n1 - 0.5, color="grey", ls="--", lw=1)
    ax.text(n1 - 0.4, ax.get_ylim()[0], " fine-tune", fontsize=8, color="grey", va="bottom")
    ax.set_title(title)
    ax.set_xlabel("epoch")
    ax.legend()
    ax.grid(alpha=0.3)
axes[0].axhline(0.85, color="green", ls=":", lw=1)  # the 85% target
fig.suptitle("Saans X-ray DenseNet121 - training history (phase 1 | phase 2)")
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()
print("saved: training_curves.png, confusion_matrix.png")

## BLOCK 5 - Save the trained weights

Saves the full model (architecture + weights) as `saans_xray_densenet121.h5`, then
reloads it from disk and re-scores the validation set to prove the file is intact.
A metrics sidecar json rides along so the backend knows exactly what it is loading.

In [ ]:
# BLOCK 5 - SAVE
import json

WEIGHTS_PATH = "saans_xray_densenet121.h5"

model.save(WEIGHTS_PATH)                                   # legacy HDF5, full model
model.save_weights("saans_xray_densenet121.weights.h5")    # weights-only spare copy
print(f"saved {WEIGHTS_PATH}  ({os.path.getsize(WEIGHTS_PATH) / 1e6:.1f} MB)")

# Round-trip check: reload from disk, re-run the validation set, compare.
reloaded = tf.keras.models.load_model(WEIGHTS_PATH)
reloaded_prob = reloaded.predict(val_ds, verbose=0).ravel()
max_drift = float(np.max(np.abs(reloaded_prob - y_prob)))
print(f"reload check - max probability drift vs the in-memory model: {max_drift:.2e}")
assert max_drift < 1e-4, "the reloaded model does not match the trained one"
print("reload OK: the .h5 on disk reproduces the model you just trained.")

metadata = {
    "model": "densenet121",
    "framework": f"tensorflow-{tf.__version__}",
    "input_shape": [IMG_SIZE, IMG_SIZE, 3],
    "preprocessing": "tf.keras.applications.densenet.preprocess_input (torch mode)",
    "output": "single sigmoid unit, P(TB)",
    "classes": {"0": "Normal", "1": "TB"},
    "threshold": THRESHOLD,
    "gradcam_layer": "conv5_block16_concat",
    "train_datasets": ["NLM Shenzhen (ChinaSet)", "NLM Montgomery County"],
    "n_train": int(len(y_train)),
    "n_val": int(len(y_val)),
    "val_metrics": {
        "accuracy": round(float(acc), 4),
        "precision": round(float(prec), 4),
        "recall": round(float(rec), 4),
        "roc_auc": round(float(auc), 4),
        "confusion_matrix": cm.tolist(),
    },
    "scope_note": (
        "Trained on adult TB CXR datasets as proof-of-concept; paediatric fine-tuning "
        "is the Phase-1 pilot roadmap item."
    ),
}
with open("saans_xray_densenet121.json", "w") as fh:
    json.dump(metadata, fh, indent=2)
print(json.dumps(metadata, indent=2))

## BLOCK 6 - Grad-CAM

Real Grad-CAM, not a stand-in: gradients of the TB **logit** with respect to the
last convolutional feature map (`conv5_block16_concat`, 7x7x1024), pooled
channel-wise into weights, weighted sum, ReLU, normalise, upsample, blend.

The head's sigmoid is temporarily swapped for a linear activation while taping, so
the gradients do not vanish when the model is confident.

In [ ]:
# BLOCK 6 - GRAD-CAM
LAST_CONV_LAYER = "conv5_block16_concat"
_GRAD_MODELS = {}


def _grad_model_for(model, last_conv_layer_name):
    key = (id(model), last_conv_layer_name)
    if key not in _GRAD_MODELS:
        _GRAD_MODELS[key] = tf.keras.models.Model(
            model.inputs, [model.get_layer(last_conv_layer_name).output, model.output]
        )
    return _GRAD_MODELS[key]


def make_gradcam_heatmap(preprocessed_batch, model=None, last_conv_layer_name=LAST_CONV_LAYER):
    """
    Real Grad-CAM. `preprocessed_batch` is (1, 224, 224, 3), already through
    densenet preprocess_input. Returns a (7, 7) float32 map normalised to [0, 1].
    """
    model = model if model is not None else globals()["model"]
    head = model.get_layer("tb_output")
    saved_activation = head.activation
    head.activation = tf.keras.activations.linear  # tape the logit, not the squashed probability
    try:
        grad_model = _grad_model_for(model, last_conv_layer_name)
        with tf.GradientTape() as tape:
            conv_out, logit = grad_model(preprocessed_batch, training=False)
            score = logit[:, 0]
        grads = tape.gradient(score, conv_out)
        if grads is None:
            raise RuntimeError("no gradient reached the target conv layer")
        weights = tf.reduce_mean(grads, axis=(1, 2))              # (1, C)
        cam = tf.einsum("bhwc,bc->bhw", conv_out, weights)[0]     # (7, 7)
        cam = tf.nn.relu(cam)
        cam = cam / (tf.reduce_max(cam) + 1e-8)
    finally:
        head.activation = saved_activation
    return cam.numpy()


def overlay_heatmap(pil_image, cam, alpha=0.45, size=(512, 512)):
    """Blend a [0,1] Grad-CAM map over the original film with a jet colour ramp."""
    base_arr = np.asarray(pil_image.convert("RGB").resize(size, Image.BICUBIC)).astype("float32") / 255.0

    cam_up = np.asarray(
        Image.fromarray(np.uint8(np.clip(cam, 0, 1) * 255)).resize(size, Image.BICUBIC)
    ).astype("float32") / 255.0

    heat = plt.get_cmap("jet")(cam_up)[..., :3]
    mask = (cam_up ** 1.5)[..., None] * alpha  # fade the cold regions so the film stays readable
    blended = base_arr * (1.0 - mask) + heat * mask
    return Image.fromarray(np.uint8(np.clip(blended, 0, 1) * 255))


print("Grad-CAM target layer:", model.get_layer(LAST_CONV_LAYER).output.shape)

In [ ]:
# BLOCK 6 (cont.) - run Grad-CAM on real validation films.
def load_for_model(image):
    """Path / PIL image -> (preprocessed (1,224,224,3) batch, original PIL image)."""
    if isinstance(image, (str, os.PathLike, Path)):
        image = Image.open(image)
    image.load()
    arr = np.asarray(
        image.convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR), dtype="float32"
    )[None, ...]
    return preprocess_input(arr.copy()), image


picks = list(df_val.index[(y_true == 1) & (y_pred == 1)][:2]) + list(
    df_val.index[(y_true == 0) & (y_pred == 0)][:1]
)
if not picks:  # nothing classified correctly - still show what the model looks at
    picks = list(df_val.index[:3])

fig, axes = plt.subplots(len(picks), 2, figsize=(9, 4.5 * len(picks)))
axes = np.atleast_2d(axes)
for row, i in enumerate(picks):
    path = df_val.loc[i, "path"]
    batch_in, original = load_for_model(path)
    cam = make_gradcam_heatmap(batch_in)
    axes[row, 0].imshow(original.convert("L"), cmap="gray")
    axes[row, 0].set_title(f"{Path(path).name}\ntrue = {CLASS_NAMES[y_true[i]]}")
    axes[row, 1].imshow(overlay_heatmap(original, cam))
    axes[row, 1].set_title(f"Grad-CAM - P(TB) = {y_prob[i]:.3f}")
    for ax in axes[row]:
        ax.axis("off")
plt.tight_layout()
plt.show()

## BLOCK 7 - Inference

`predict_xray(image)` takes a path, a PIL image, or raw bytes and returns

```python
{"tb_probability": 0.9137, "heatmap_image": <PIL.Image 512x512>}
```

`heatmap_to_data_url()` is the one extra step the FastAPI layer in
`server/vision.py` needs - it produces the same `data:image/png;base64,...` string
the frontend already consumes.

In [ ]:
# BLOCK 7 - INFERENCE
# Trained on adult TB CXR datasets as proof-of-concept; paediatric fine-tuning is
# the Phase-1 pilot roadmap item. Every output here is decision support for a
# health worker, never a diagnosis.
import base64
import io


def predict_xray(image, model=None, threshold=THRESHOLD, alpha=0.45):
    """
    New chest X-ray in -> TB probability + Grad-CAM overlay out.

    image: file path, PIL.Image, or raw image bytes.
    returns: {"tb_probability": float, "heatmap_image": PIL.Image}
    """
    model = model if model is not None else globals()["model"]
    if isinstance(image, (bytes, bytearray)):
        image = Image.open(io.BytesIO(image))

    batch_in, original = load_for_model(image)
    probability = float(model.predict(batch_in, verbose=0)[0][0])
    cam = make_gradcam_heatmap(batch_in, model)

    return {
        "tb_probability": round(probability, 4),
        "heatmap_image": overlay_heatmap(original, cam, alpha=alpha),
    }


def heatmap_to_data_url(pil_image):
    """PNG data URL, ready for the Saans API response / an <img src>."""
    buf = io.BytesIO()
    pil_image.save(buf, format="PNG", optimize=True)
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode("ascii")


# --- run it on a real held-out film -----------------------------------------
sample_path = df_val.loc[picks[0], "path"]
result = predict_xray(sample_path)

print("input :", Path(sample_path).name, f"(true label: {CLASS_NAMES[int(df_val.loc[picks[0], 'label'])]})")
print("output: {'tb_probability':", result["tb_probability"], ", 'heatmap_image':", result["heatmap_image"], "}")
print("verdict:", "TB-suspect" if result["tb_probability"] >= THRESHOLD else "no TB features detected")
print("data url:", len(heatmap_to_data_url(result["heatmap_image"])), "chars")

plt.figure(figsize=(5, 5))
plt.imshow(result["heatmap_image"])
plt.axis("off")
plt.title(f"predict_xray -> P(TB) = {result['tb_probability']:.3f}")
plt.show()

In [ ]:
# BLOCK 7 (cont.) - sweep the whole validation split through the public
# inference function, so what ships is what was measured.
sweep = np.array([predict_xray(p)["tb_probability"] for p in tqdm(df_val["path"], desc="predict_xray")])
sweep_acc = accuracy_score(y_true, (sweep >= THRESHOLD).astype(int))
print(f"accuracy through predict_xray(): {sweep_acc:.4f}   (block 4 reported {acc:.4f})")

## BLOCK 8 - Download the weights

Run the cell below: your browser downloads `saans_xray_densenet121.h5` (plus the
metrics json and the plots). Allow multiple downloads if Chrome asks.

Then move the `.h5` into the repo at:

```
Saans/backend/models/saans_xray_densenet121.h5
```

If the download is blocked, use the **folder icon** in Colab's left sidebar, find
`saans_xray_densenet121.h5` under `/content`, click the three dots next to it and
choose **Download**.

**Backend note:** `server/vision.py` currently loads *PyTorch* weights
(`densenet121_tb.pt`, via `torchvision`). This notebook produces a *Keras* `.h5`,
so `load_model()` / `run_model()` there need swapping to
`tf.keras.models.load_model(...)` plus the Grad-CAM from Block 6 - those functions
are written to drop straight in, and `heatmap_to_data_url` matches the response
format the API already returns.

In [ ]:
# BLOCK 8 - download to your machine
from google.colab import files

for name in [
    "saans_xray_densenet121.h5",
    "saans_xray_densenet121.json",
    "training_curves.png",
    "confusion_matrix.png",
]:
    if os.path.exists(name):
        print(f"downloading {name} ({os.path.getsize(name) / 1e6:.1f} MB)")
        files.download(name)
    else:
        print(f"missing: {name} - run the earlier blocks first")